# Trying the job extraction pipeline

The pipeline reads a messy freelance job posting and fills in a fixed form: project type, tools, experience level,
deadline, budget and red flags. Then `verify.py` checks every claim against the posting's own text.

Here we use it, step by step: look at real postings and what the model made of them, see the check catch a bad
quote, then send new postings through the live model.

Steps 1 to 4, 9 and 10 read the real postings. The dataset is not included in this repository, so those cells show no output here. Run them against your own copy.

## Setup
Load the cleaned postings and the extractions we already saved from the full run.

In [1]:
import json
import os
from pathlib import Path

# The project reads .env and data/ relative to the project folder, so start there.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from job_extraction.extract import Extractor, Settings
from job_extraction.schema import Evidence, JobExtraction
from job_extraction.verify import check


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()]


postings = {p["id"]: p for p in read_jsonl("data/postings.jsonl")}
saved = {r["id"]: r for r in read_jsonl("data/extractions.jsonl")}
print(len(postings), "postings,", len(saved), "already extracted")


def show(data):
    """Print an extraction as a short list."""
    for field, value in data.model_dump(mode="json").items():
        if field == "evidence":
            for item in value:
                print(f"  quote for {item['field']}: \"{item['quote']}\"")
        else:
            print(f"{field:>17}: {value}")

8595 postings, 8595 already extracted


## Step 1: A real posting
This is what the model receives: a title and some free text.

In [ ]:
posting_id = "37362477"
posting = postings[posting_id]
print(posting["title"])
print()
print(posting["description"])

## Step 2: What the model extracted
This was saved from the full run. The model was forced to answer in the shape of `JobExtraction`, so every field is
typed. Each fact that is easy to invent comes with a quote.

In [ ]:
extraction = JobExtraction.model_validate(saved[posting_id]["data"])
show(extraction)

## Step 3: Check it against the posting
`check()` makes no model call. It looks for every quote in the posting text, and makes sure a claimed fact has a quote.
An empty list means the extraction passed.

In [4]:
text = posting["title"] + " " + posting["description"]
print("problems:", check(extraction, text))

problems: []


## Step 4: A posting that failed the check
Of the 8,595 postings, 162 had at least one problem. Here is one. The quote is supposed to be copied exactly.

In [ ]:
bad_id = "37326518"
bad_text = postings[bad_id]["title"] + " " + postings[bad_id]["description"]
bad = JobExtraction.model_validate(saved[bad_id]["data"])

print("end of the posting:", bad_text[-90:])
print("the model's quote: ", bad.evidence[0].quote)
print()
print("problems:", check(bad, bad_text))

The model added a full stop after "James" that the posting does not have, so the strict check rejects it.
The claim itself is reasonable. The check can prove a quote is real, not that the model understood it.

## Step 5: Send new postings through the live model
These three postings are invented, not client text. This step calls the model, which uses a few thousand tokens in
total. The first is detailed, the second is risky, and the third says almost nothing.

In [6]:
extractor = Extractor(Settings())

new_postings = {
    "detailed": ("Tableau dashboard from Excel sales data",
                 ("I need a Tableau dashboard built from my Excel file of monthly sales. It must be finished "
                  "within 2 weeks. My budget is $300. An intermediate level freelancer is fine.")),
    "risky": ("Data help needed",
              ("Please send me a free sample of your work first. After that contact me on WhatsApp to discuss. "
               "I need some data stuff done.")),
    "sparse": ("Clean a CSV",
               "Looking for someone to clean a CSV."),
}

results = {}
for name, (title, description) in new_postings.items():
    data, tokens = await extractor.extract(title, description)
    results[name] = (data, title + " " + description)
    print(f"--- {name} ({tokens} tokens)")
    show(data)
    print("  problems:", check(data, results[name][1]))
    print()

--- detailed (832 tokens)
     project_type: dashboard_bi
     deliverables: ['Tableau dashboard']
            tools: ['tableau', 'excel']
 experience_level: intermediate
    timeline_days: 14
        is_urgent: False
       is_ongoing: False
    stated_budget: $300
        red_flags: []
  quote for experience_level: "An intermediate level freelancer is fine."
  quote for timeline_days: "It must be finished within 2 weeks."
  quote for stated_budget: "My budget is $300."
  problems: []



--- risky (828 tokens)
     project_type: other
     deliverables: []
            tools: []
 experience_level: not_stated
    timeline_days: None
        is_urgent: False
       is_ongoing: False
    stated_budget: None
        red_flags: ['off_platform_contact', 'free_work_requested', 'vague_brief']
  quote for red_flags: "Please send me a free sample of your work first."
  quote for red_flags: "After that contact me on WhatsApp to discuss."
  quote for red_flags: "I need some data stuff done."
  problems: []



--- sparse (771 tokens)
     project_type: data_analysis
     deliverables: ['cleaned CSV']
            tools: ['csv']
 experience_level: not_stated
    timeline_days: None
        is_urgent: False
       is_ongoing: False
    stated_budget: None
        red_flags: ['vague_brief']
  quote for red_flags: "Looking for someone to clean a CSV."
  problems: []



Look for three things:

- **detailed** has a budget of `$300`, a timeline of `14` days, and a quote for each.
- **risky** should show `off_platform_contact` and `free_work_requested`, each with a quote.
- **sparse** should be mostly `null`, `not_stated` and empty lists. Not guessing is the point.

## Step 6: Does the check catch a made-up fact?
Take the detailed extraction and change it by hand to claim a budget the posting never mentioned.

In [7]:
data, text = results["detailed"]

# Keep the real quotes, but swap the budget quote for one the posting never contained.
kept = [item for item in data.evidence if item.field != "stated_budget"]
faked = data.model_copy(update={
    "stated_budget": "$500",
    "evidence": kept + [Evidence(field="stated_budget", quote="I will pay $500")],
})

print("original:", check(data, text))
print("faked:   ", check(faked, text))

original: []
faked:    ['quote for stated_budget is not in the posting', 'stated_budget numbers are not in the posting']


## Step 7: Try your own
Paste a posting below and run the cell.

In [8]:
title = "Need a Python script to scrape product prices"
description = "Looking for an expert to scrape prices from 5 online shops every day. Budget around 150 euros."

data, tokens = await extractor.extract(title, description)
show(data)
print("problems:", check(data, title + " " + description))

     project_type: web_scraping
     deliverables: ['Python script', 'scraped product prices']
            tools: ['python']
 experience_level: expert
    timeline_days: None
        is_urgent: False
       is_ongoing: True
    stated_budget: 150 euros
        red_flags: []
  quote for experience_level: "Looking for an expert to scrape prices from 5 online shops every day."
  quote for stated_budget: "Budget around 150 euros."
problems: []


## Step 8: The whole run in numbers
Counts across all 8,595 postings. No posting text is shown here.

In [9]:
import pandas as pd

rows = [{"problems": r["problems"], **r["data"]} for r in saved.values()]
df = pd.DataFrame(rows)

print(f"passed every check: {(df['problems'].str.len() == 0).mean():.1%}")
print(f"stated a budget:    {df['stated_budget'].notna().mean():.1%}")
print(f"flagged urgent:     {df['is_urgent'].mean():.1%}")
print()
print(df["project_type"].value_counts().to_string())
print()
print("red flags:")
print(df["red_flags"].explode().value_counts().to_string())

passed every check: 97.1%
stated a budget:    3.1%
flagged urgent:     10.2%

project_type
data_analysis       4164
software_dev        1305
machine_learning    1130
dashboard_bi         798
web_scraping         335
writing_research     315
data_entry           311
other                237

red flags:
red_flags
vague_brief             751
off_platform_contact    295
free_work_requested      68


## Step 9: Ask questions about the postings (RAG)
So far the model reads one posting at a time. RAG lets you ask a question about all of them. Every posting was embedded
once (`uv run jobs index`). To answer a question we embed it too, find the 8 postings closest in meaning, and give them
to the model. It answers from those postings only, and cites them by number.

In [ ]:
from job_extraction.ask import Asker

asker = Asker(Path("data/postings.jsonl"), Path("data/extractions.jsonl"))

question = "What do clients ask for when they want a web scraping job, and how soon do they need it?"
text, found = asker.ask(question)

print(text)
print()
for number, posting in enumerate(found, 1):
    print(f"[{number}] {posting['id']}  {posting['title'][:80]}")

## Step 10: Check a citation yourself
The answer cites postings by number. Pick one and read what it really says.

In [ ]:
number = 1
print(found[number - 1]["title"])
print()
print(found[number - 1]["description"][:600])